# SuAVE Notebook Dispatcher

**Binder**: survey parameters were loaded automatically — just run all cells.  
**Colab**: paste the session token SuAVE copied to your clipboard into `SUAVE_TOKEN`,
enter your SuAVE server URL in `SUAVE_HOST`, then run all cells.

After loading, click any notebook card below to open it in a new tab.
Parameters are shared automatically within the same session — no token needed again.

In [ ]:
# ── Cell 1: Load SuAVE parameters ─────────────────────────────────────────
import sys
sys.path.insert(0, 'helpers')
import suave_utils as su
from IPython.display import display, HTML

# ---- Colab only: paste values from the SuAVE launch dialog ---------------
SUAVE_TOKEN = ''   # @param {type:"string"}
SUAVE_HOST  = ''   # @param {type:"string"}  e.g. https://suave.sdsc.edu
# --------------------------------------------------------------------------

params = su.load_params(token=SUAVE_TOKEN, host=SUAVE_HOST)

if params:
    su.show_params(params)
else:
    display(HTML(
        '<p style="color:#e74c3c;font-weight:bold">'
        'Parameters not found. On Colab, enter SUAVE_TOKEN and SUAVE_HOST above '
        'and re-run this cell.'
        '</p>'
    ))

In [ ]:
# ── Cell 2: Choose an operation ────────────────────────────────────────────
from IPython.display import display, HTML

if params is None:
    raise RuntimeError('Run Cell 1 first.')

# ── Detect what this survey and hub support ──────────────────────────────────
caps             = su.detect_capabilities(params)
has_images       = caps['has_images']
has_netvis       = caps['has_netvis']
has_largedataset = caps['has_largedataset']
localdzc         = caps['localdzc']
full_images      = caps['full_images']

# ── Menu definition ──────────────────────────────────────────────────────────
# Availability keys:
#   'any'         — always shown
#   'image'       — full-size images accessible on NFS (/lib-nfs/dzgen/...)
#   'netvis'      — survey has a #netvis network data file
#   'largedataset'— /lib-nfs/largedatasets is mounted on this hub

SECTIONS = [
    ('Data \u0026 Statistics', '\U0001f4ca', '#6366f1', [
        ('Descriptive Statistics',      'operations/stats/DescriptiveStats.ipynb',              'any'),
        ('Contingency Tables',          'operations/stats/Generate_Contingency_Tables.ipynb',   'any'),
        ('Factor Contributions',        'operations/stats/Generate_Factor_Contributions.ipynb', 'any'),
        ('Arithmetic Operations',       'operations/arithmetic/SuaveArithmetic.ipynb',          'any'),
        ('Spatial Statistics',          'operations/spatialstats/SpatialStats.ipynb',           'any'),
        ('Generate Maps',               'operations/maps/Generate_Aggregate_Maps_Suave.ipynb',  'any'),
        ('Network Analysis',            'operations/networks/networks.ipynb',                   'netvis'),
    ]),
    ('Text \u0026 NLP', '\U0001f4dd', '#0ea5e9', [
        ('Named Entity Recognition',    'operations/tagger/NER.ipynb',                         'any'),
        ('SDG Dataset',                 'operations/SDG/GenerateSDGDataset.ipynb',              'largedataset'),
    ]),
    ('Data Wrangling', '\U0001f527', '#10b981', [
        ('Enhance Dataset',             'operations/wrangling/qualgeoimage.ipynb',              'any'),
        ('Explore with Holoviz',        'operations/holoviz/holoviz.ipynb',                    'any'),
    ]),
    ('Image Analysis', '\U0001f5bc\ufe0f', '#f59e0b', [
        ('Color Statistics',            'operations/colors/ColorStats.ipynb',                  'image'),
        ('Classify Images (Clarifai)',  'operations/classify/ImageClassify.ipynb',             'image'),
        ('LeNet CNN Model',             'operations/predict/PredictiveModel_v2.ipynb',         'image'),
        ('Extend LeNet Model',          'operations/predict/ExtendModel.ipynb',                'image'),
        ('SVM Predictive Model',        'operations/svm/SVMPredictiveModel.ipynb',             'image'),
        ('Transfer Learning',           'operations/transfer_learning/transfer_learning.ipynb','image'),
    ]),
]

def _avail(kind):
    return (
        kind == 'any'
        or (kind == 'image'        and has_images)
        or (kind == 'netvis'       and has_netvis)
        or (kind == 'largedataset' and has_largedataset)
    )

# ── Build warning banners ────────────────────────────────────────────────────
import os
warnings = []
if localdzc and os.path.isfile(localdzc) and not os.path.isdir(full_images):
    warnings.append(
        'This hub supports image-based processing, but full-size images have not '
        'been generated for this survey. Contact the admin to re-generate images '
        'from image tiles. Image analysis operations are unavailable.'
    )
elif not localdzc:
    warnings.append(
        'Image tiles are not on NFS-mounted storage for this survey '
        '(or this hub does not support NFS). Image analysis operations are unavailable.'
    )
if not has_largedataset:
    warnings.append(
        'Large external datasets (e.g. SDG) are not mounted on this hub '
        '(/lib-nfs/largedatasets). Those operations are unavailable.'
    )

# ── Build HTML card menu ─────────────────────────────────────────────────────
parts = ['''
<style>
  .sd-wrap { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
              max-width: 780px; padding: 6px 0 12px; }
  .sd-warn { padding: 9px 13px; border-radius: 6px; font-size: 12px; line-height: 1.5;
              margin-bottom: 10px; background: #fffbeb; border: 1px solid #f59e0b;
              color: #92400e; }
  .sd-section { margin: 18px 0 6px; }
  .sd-section-hd { display: flex; align-items: center; gap: 7px;
                    font-size: 11px; font-weight: 700; text-transform: uppercase;
                    letter-spacing: .07em; margin-bottom: 9px;
                    padding-bottom: 6px; border-bottom: 2px solid; }
  .sd-cards { display: flex; flex-wrap: wrap; gap: 7px; }
  .sd-card { display: inline-block; padding: 7px 14px; border-radius: 7px;
              border: 1.5px solid; font-size: 13px; font-weight: 500;
              text-decoration: none !important; line-height: 1.3;
              transition: filter .12s, transform .12s; }
  .sd-card:hover { filter: brightness(1.18); transform: translateY(-1px);
                    text-decoration: none !important; }
  .sd-note { margin-top: 20px; font-size: 11px; color: #6b7280; line-height: 1.7;
              padding: 10px 14px; border-radius: 6px; background: #f9fafb;
              border: 1px solid #e5e7eb; }
</style>
<div class="sd-wrap">
''']

for w in warnings:
    parts.append(f'<div class="sd-warn">&#9888;&nbsp; {w}</div>')

any_section = False
for title, icon, color, items in SECTIONS:
    visible = [(lbl, path) for lbl, path, kind in items if _avail(kind)]
    if not visible:
        continue
    any_section = True
    bg     = color + '18'
    border = color + '60'
    hdbdr  = color + '50'
    parts.append(
        f'<div class="sd-section">'
        f'<div class="sd-section-hd" style="color:{color};border-color:{hdbdr}">'
        f'<span>{icon}</span><span>{title}</span></div>'
        f'<div class="sd-cards">'
    )
    for lbl, path in visible:
        url = su.make_nb_url(path)
        parts.append(
            f'<a class="sd-card" href="{url}" target="_blank" '
            f'style="color:{color};background:{bg};border-color:{border}">'
            f'{lbl}</a>'
        )
    parts.append('</div></div>')

if not any_section:
    parts.append('<p style="color:#6b7280">No operations available for this survey.</p>')

colab_note = (
    ' On Colab, open the new notebook inside the <em>same runtime session</em> '
    '(File \u2192 Open notebook) so it shares the loaded parameters.'
    if su.in_colab() else ''
)
parts.append(
    f'<div class="sd-note">'
    f'\u2197\ufe0f Each notebook opens in a new tab and reads the same survey '
    f'parameters automatically \u2014 no token required.{colab_note}'
    f'</div>'
)
parts.append('</div>')

display(HTML(''.join(parts)))